# Week 42, Day 5: Cycle detection

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w042d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

a, b, c, d = Node("a"), Node("b"), Node("c"), Node("d")
a.next, b.next, c.next, d.next = b, c, d, b       # d points back to b

out, node = [], a
while node is not None and len(out) < 10:
    out.append(node.value)
    node = node.next
print("".join(out))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def first_repeat(head):
    seen = set()
    node = head
    while node is not None:
        if id(node) in seen:
            return node
        seen.add(id(node))
        node = node.next
    return None

a, b, c, d = Node("a"), Node("b"), Node("c"), Node("d")
a.next, b.next, c.next, d.next = b, c, d, b
print(first_repeat(a).value)
d.next = None
print(first_repeat(a))
twins = Node(1, Node(1, Node(1)))                  # repeated values, no cycle
print(first_repeat(twins))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def has_cycle(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

nodes = [Node(i) for i in range(6)]
for x, y in zip(nodes, nodes[1:]):
    x.next = y
print(has_cycle(nodes[0]))
nodes[-1].next = nodes[2]
print(has_cycle(nodes[0]))
print(has_cycle(None), has_cycle(Node(0)))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def cycle_start(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            break
    else:
        return None                          # the loop ended normally: no cycle
    slow = head
    while slow is not fast:
        slow, fast = slow.next, fast.next
    return slow

nodes = [Node(c) for c in "abcdefg"]
for x, y in zip(nodes, nodes[1:]):
    x.next = y
nodes[-1].next = nodes[3]                    # g -> d
print(cycle_start(nodes[0]).value)

In [ ]:
def step(n):
    return sum(int(d) ** 2 for d in str(n))

def is_happy(n):
    slow = fast = n
    while True:
        slow = step(slow)
        fast = step(step(fast))
        if slow == fast:                     # values, this time: the "nodes" are numbers
            return slow == 1

print([n for n in range(1, 50) if is_happy(n)])

> **Key idea.** A cycle makes a list loop for ever; detect it before trusting the list. A set of visited node `id`s finds the first repeated node in O(n) time and O(n) space. Floyd's tortoise (one step) and hare (two steps) need O(1) space: if the hare reaches `None` there's no cycle; otherwise the gap between them closes by one each step, so they meet. Move, then compare. To find the cycle's start, reset one pointer to the head and step both one at a time until they meet. Compare nodes by identity, not value. The same method works on any sequence `x, f(x), f(f(x)), ...`: happy numbers, and finding a duplicate in a read-only list.

### Exercise: A set of visited nodes

Write `first_repeat(head)`: walk the list keeping a set of the `id`s of nodes seen, and return the first node visited twice (the start of the cycle), or `None` if the list ends. Nodes with equal values are **different** nodes.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def first_repeat(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def chain(values, back=None):\n    nodes = [Node(v) for v in values]\n    for x, y in zip(nodes, nodes[1:]):\n        x.next = y\n    if back is not None and nodes:\n        nodes[-1].next = nodes[back]\n    return nodes\ndef test_repeat():\n    \"repeats\"\n    n = chain(\"abcde\", back=2)\n    assert first_repeat(n[0]) is n[2]\n    n = chain(\"aaaa\")\n    assert first_repeat(n[0]) is None, \"equal values aren't a cycle\"\n    n = chain(\"x\", back=0)\n    assert first_repeat(n[0]) is n[0] and first_repeat(None) is None")

### Exercise: Tortoise and hare

Write `has_cycle(head)` with Floyd's method: two pointers, one moving one step and one moving two, no sets, lists or dictionaries. Return `True` if there's a cycle.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def has_cycle(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\ndef chain(n, back=None):\n    nodes = [Node(i) for i in range(n)]\n    for x, y in zip(nodes, nodes[1:]):\n        x.next = y\n    if back is not None and nodes:\n        nodes[-1].next = nodes[back]\n    return nodes[0] if nodes else None\ndef test_floyd():\n    \"cycles\"\n    assert not has_cycle(None) and not has_cycle(chain(1)) and not has_cycle(chain(2)) and not has_cycle(chain(7))\n    assert has_cycle(chain(1, 0)) and has_cycle(chain(2, 0)) and has_cycle(chain(2, 1)) and has_cycle(chain(9, 4))\n    assert not has_cycle(chain(50_000)) and has_cycle(chain(50_000, 1))\n    tree = ast.parse(source)\n    assert not any(isinstance(n, (ast.Set, ast.Dict, ast.List, ast.SetComp, ast.DictComp, ast.ListComp)) for n in ast.walk(tree))\n    assert not any(isinstance(n, ast.Call) and getattr(n.func, \"id\", \"\") in (\"set\", \"dict\", \"list\") for n in ast.walk(tree)), \"O(1) space\"")

### Exercise: Where it starts, and how long

Write `cycle_info(head)`: return `(start, length)` for a list with a cycle, where `start` is the cycle's first node and `length` the number of nodes in it; `(None, 0)` if there's no cycle. Use Floyd's method to meet, reset one pointer to the head to find the start, then walk once round to count. O(1) space.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def cycle_info(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef chain(n, back=None):\n    nodes = [Node(i) for i in range(n)]\n    for x, y in zip(nodes, nodes[1:]):\n        x.next = y\n    if back is not None and nodes:\n        nodes[-1].next = nodes[back]\n    return nodes\ndef test_info():\n    \"start and length\"\n    rng = random.Random(8)\n    for _ in range(200):\n        n = rng.randint(1, 30)\n        if rng.random() < 0.2:\n            nodes = chain(n)\n            assert cycle_info(nodes[0]) == (None, 0)\n        else:\n            back = rng.randint(0, n - 1)\n            nodes = chain(n, back)\n            start, length = cycle_info(nodes[0])\n            assert start is nodes[back] and length == n - back, (n, back)\n    assert cycle_info(None) == (None, 0)")

### Exercise: A safe length

Write `safe_length(head)`: the number of **distinct** nodes in the list, whether it ends or loops. For a list with a cycle, that's the tail plus the cycle. Use `cycle_info` from the starter (it's correct), and O(1) space.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def cycle_info(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            break
    else:
        return None, 0
    slow = head
    while slow is not fast:
        slow, fast = slow.next, fast.next
    length, node = 1, slow.next
    while node is not slow:
        length += 1
        node = node.next
    return slow, length

def safe_length(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def chain(n, back=None):\n    nodes = [Node(i) for i in range(n)]\n    for x, y in zip(nodes, nodes[1:]):\n        x.next = y\n    if back is not None and nodes:\n        nodes[-1].next = nodes[back]\n    return nodes[0] if nodes else None\ndef test_len():\n    \"lengths\"\n    assert safe_length(None) == 0 and safe_length(chain(5)) == 5 and safe_length(chain(5, 0)) == 5 and safe_length(chain(5, 3)) == 5 and safe_length(chain(1, 0)) == 1\n    assert safe_length(chain(20_000, 7)) == 20_000")

### Exercise: Happy numbers

Write `digit_squares(n)` (the sum of the squares of `n`'s digits) and `is_happy(n)`: does repeating `digit_squares` from `n` reach 1? Use the tortoise and hare on the sequence (no sets). Then set `first_ten` to the first ten happy numbers.

In [ ]:
def digit_squares(n):
    ___

def is_happy(n):
    ___

first_ten = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_happy():\n    \"happy numbers\"\n    assert digit_squares(19) == 82 and digit_squares(100) == 1 and digit_squares(0) == 0\n    assert is_happy(19) and is_happy(1) and is_happy(7) and not is_happy(2) and not is_happy(4)\n    assert first_ten == [1, 7, 10, 13, 19, 23, 28, 31, 32, 44]\n    assert \"set(\" not in source")

### Exercise: The duplicate, without changing anything

`nums` has `n + 1` numbers, each from 1 to `n`, so at least one value repeats; exactly one value appears more than once (possibly many times). Write `find_duplicate(nums)`, which returns it in O(n) time and **O(1)** extra space, **without changing** `nums` and without sorting. Treat position `i` as a node whose `next` is position `nums[i]`, start at position 0, and find where the cycle starts.

In [ ]:
def find_duplicate(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\nimport random\ndef test_dup():\n    \"duplicates\"\n    assert find_duplicate([1, 3, 4, 2, 2]) == 2 and find_duplicate([3, 1, 3, 4, 2]) == 3 and find_duplicate([1, 1]) == 1 and find_duplicate([2, 2, 2, 2, 2]) == 2\n    rng = random.Random(9)\n    for _ in range(100):\n        n = rng.randint(1, 50)\n        dup = rng.randint(1, n)\n        others = [x for x in range(1, n + 1) if x != dup]\n        keep = rng.sample(others, rng.randint(0, len(others)))\n        nums = keep + [dup] * (n + 1 - len(keep))\n        rng.shuffle(nums)\n        before = list(nums)\n        assert find_duplicate(nums) == dup and nums == before, (nums, dup)\n    tree = ast.parse(source)\n    assert not any(isinstance(n, ast.Call) and getattr(n.func, \"id\", getattr(n.func, \"attr\", \"\")) in (\"set\", \"sorted\", \"sort\", \"Counter\", \"dict\") for n in ast.walk(tree)), \"O(1) space, no sorting\"")

### Bug hunt: The hare trips at the end

This moves the hare two steps without checking that the first step exists, so on a list with an **odd** number of nodes and no cycle it crashes with `AttributeError: 'NoneType' object has no attribute 'next'`. Fix the loop condition so this prints `False True`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def has_cycle(head):
    slow = fast = head
    while fast is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

a = Node(1, Node(2, Node(3)))
b = Node(1, Node(2))
b.next.next = b
print(has_cycle(a), has_cycle(b))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check fast.next too\"\n    expect_output(\"False True\")\n    assert has_cycle(Node(1)) is False and has_cycle(None) is False")

### Bug hunt: Same value, different node

This remembers the **values** it has seen rather than the nodes, so a list that merely repeats a value is reported as having a cycle: this prints `True`. Remember the nodes, so it prints `False`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def has_cycle(head):
    seen = set()
    node = head
    while node is not None:
        if node.value in seen:
            return True
        seen.add(node.value)
        node = node.next
    return False

print(has_cycle(Node("hi", Node("there", Node("hi")))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"nodes, not values\"\n    expect_output(\"False\")\n    a = Node(1)\n    a.next = a\n    assert has_cycle(a)")

### Bug hunt: Caught before the race begins

This compares the two pointers **before** moving them. Both start at the head, so every list is reported as having a cycle: this prints `True`. Move first, then compare, so it prints `False`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def has_cycle(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        if slow is fast:
            return True
        slow = slow.next
        fast = fast.next.next
    return False

print(has_cycle(Node(1, Node(2, Node(3, Node(4))))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"move, then compare\"\n    expect_output(\"False\")\n    a = Node(1, Node(2))\n    a.next.next = a\n    assert has_cycle(a)")

### Puzzle: Finding the start

Arrange the second half of `cycle_start`: after the pointers have met, find the first node of the cycle. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code.

p = ___   # the extra space of Floyd's method: "O(1)" or "O(n)"
q = ___   # by how much the gap between hare and tortoise closes each step inside the cycle
r = ___   # the sequence from 4 under digit_squares: 4, 16, 37, 58, 89, 145, 42, 20, then?
s = ___   # True or False: two nodes holding equal values mean the list has a cycle
t = ___   # in [2, 3, 1, 3], following i -> nums[i] from position 0, the first position visited twice

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"O(1)\" and q == 1 and r == 4, \"p, q and r\"\n    assert s is False and t == 3, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: repairing and joining lists

Two more classic problems, both in O(1) extra space:

1. `break_cycle(head)`: if the list has a cycle, cut it, by setting the `next` of the cycle's **last** node (the one that points back to the start) to `None`, so the list becomes an ordinary list with every node in it once. Return the number of nodes. A list without a cycle is left alone.
2. `meeting_point(a, b)`: two ordinary lists (no cycles) may **join** at some node and share everything after it, like a `Y`. Return the first shared node, or `None` if they never join. Hint: walk two pointers, one along each list; when one runs off the end, send it to the start of the **other** list. After at most `len(a) + len(b)` steps, they're at the same node (or both at `None`).

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def break_cycle(head):
    ___

def meeting_point(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef chain(n, back=None):\n    nodes = [Node(i) for i in range(n)]\n    for x, y in zip(nodes, nodes[1:]):\n        x.next = y\n    if back is not None and nodes:\n        nodes[-1].next = nodes[back]\n    return nodes\ndef walk(head, limit=10_000):\n    out = []\n    while head is not None and len(out) < limit:\n        out.append(head)\n        head = head.next\n    return out\ndef test_break():\n    \"breaking cycles\"\n    rng = random.Random(10)\n    for _ in range(100):\n        n = rng.randint(1, 25)\n        back = rng.randint(0, n - 1) if rng.random() < 0.8 else None\n        nodes = chain(n, back)\n        assert break_cycle(nodes[0]) == n\n        assert walk(nodes[0]) == nodes, (n, back)\n    assert break_cycle(None) == 0\ndef test_join():\n    \"joining\"\n    rng = random.Random(11)\n    for _ in range(100):\n        shared = chain(rng.randint(0, 6))\n        a_own, b_own = chain(rng.randint(0, 6)), chain(rng.randint(0, 6))\n        join = shared[0] if shared else None\n        if a_own:\n            a_own[-1].next = join\n        if b_own:\n            b_own[-1].next = join\n        a = a_own[0] if a_own else join\n        b = b_own[0] if b_own else join\n        assert meeting_point(a, b) is join\n    x, y = chain(3), chain(4)\n    assert meeting_point(x[0], y[0]) is None")

## Warm-up

Without the `len(out) < 10` guard, that loop would never stop: after `d` comes `b` again, for ever. The list has a **cycle**, `b → c → d → b`, with a **tail** of one node, `a`, leading into it.

## When a list doesn't end

```text
a -> b -> c -> d
     ^         |
     +---------+
```

A list with a cycle has no last node: following `next` from the head eventually goes round and round the same nodes. Every function that loops `while node is not None` (counting, printing, searching for something missing) hangs. Cycles come from bugs in relinking code, or from data built elsewhere (a file of "next" ids, a chain of redirects). Defensive code (Week 37) checks for them before trusting a list.

Two facts shape every method today:

- Cycles are about **nodes**, not values. Two different nodes can hold the same value without any cycle, so compare with `is` (or `id`), never `==`.
- A list either ends in `None` or has exactly one cycle, reached after a tail of zero or more nodes: once you're in a loop, there's no way out.

## Remembering where you've been

The straightforward way: walk the list, keeping a set of every node seen. Meeting a node twice means a cycle, and the first node met twice is where the cycle starts:

It's O(n) time, but O(n) extra **space**. (`id(node)` gives a number unique to each live object. Our `Node` doesn't define `__eq__`, so a set of the nodes themselves would also work, by identity; using `id` makes the intention clear, and still works if someone adds an `__eq__` later.)

## The tortoise and the hare

Robert Floyd's method (published by Donald Knuth, who credits it to Floyd) uses two pointers and nothing else. The **tortoise** moves one node at a time; the **hare** moves two. If the list ends, the hare reaches `None` first. If there's a cycle, both end up going round it, and the hare catches the tortoise from behind:

**Why must they meet?** Once both are inside the cycle, look at the distance from the tortoise forward to the hare, going round the cycle. Each step, the hare moves 2 and the tortoise 1, so the gap **closes by exactly 1**. A gap that shrinks by one each step must reach 0: the hare can't jump over the tortoise. And it happens quickly: the tortoise is caught before it completes one lap, so the whole thing is O(n) time, O(1) space.

Notice the loop moves **then** compares. Comparing first would find `slow is fast` at the start, when both are at the head, and report a cycle in every list.

## Where the cycle starts

When the pointers meet, one more trick finds the **start** of the cycle: put one pointer back at the head, and move **both** one step at a time. They meet again exactly at the cycle's first node.

Why it works, in outline: say the tail has `t` nodes before the cycle, the cycle has length `c`, and they meet `m` steps into the cycle. The tortoise walked `t + m`; the hare walked twice that, which is also `t + m` plus some whole number of laps, `k × c`. So `t + m = k × c`, which means `t = k × c - m`: walking `t` steps from the meeting point brings you round to the cycle's start, exactly as walking `t` steps from the head does. (The `while ... else` form, from Week 6, runs `else` only if the loop ended without `break`.)

The cycle's **length** is then easy: from any node in the cycle, walk until you come back, counting.

## Cycles everywhere

The tortoise and hare work on **any** sequence where each item decides the next: `x, f(x), f(f(x)), ...`. If there are only finitely many possible values, such a sequence must eventually repeat, so it's a "linked list" whose `next` is `f`, and it ends in a cycle.

**Happy numbers.** Replace a number by the sum of the squares of its digits, again and again: 19 → 82 → 68 → 100 → 1. A number is "happy" if this reaches 1 (which then repeats for ever); otherwise it falls into a cycle that never includes 1.

**A duplicate in a list.** A list of `n + 1` numbers, each from 1 to `n`, must contain a repeat. Read it as a linked list where position `i` points to position `nums[i]`; the repeated value is a node with two arrows into it, which is exactly where a cycle starts. Floyd's method finds it in O(n) time and O(1) space, without sorting or changing the list. You'll write it in the practice.

There's a faster relative, Brent's method (Richard Brent, 1980), which moves only one pointer and teleports the other; Floyd's is the one to know first.

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a list with a tail of 3 nodes and a cycle of 4, and play the tortoise and hare by hand: where do they meet, and does resetting one to the head bring them together at the cycle's start? Tomorrow is the Forge, where this week's linked-list skills build a text editor's buffer.